**Java 24+:** Spark/Hadoop need Java 17 (older JDKs work too). Run the cell below first — it sets `JAVA_HOME` and `YELP_JAVA17_HOME` from `mise` if installed — then **restart the kernel** and run all cells from the top.

In [1]:
# Run this first if you use Java 25+. Then restart kernel and run all.
import os, subprocess
path = ""
try:
    r = subprocess.run(["mise", "where", "java@17"], capture_output=True, text=True, timeout=5)
    path = (r.stdout or "").strip()
except Exception:
    pass
if path and os.path.exists(os.path.join(path, "bin", "java")):
    os.environ["YELP_JAVA17_HOME"] = path
    os.environ["JAVA_HOME"] = path
    print("Set JAVA_HOME and YELP_JAVA17_HOME to", path, "-- restart the kernel and run all cells.")
else:
    print("Install Java 17 first: run in terminal:  mise install java@17")

Set JAVA_HOME and YELP_JAVA17_HOME to /home/dash/.local/share/mise/installs/java/17.0.2 -- restart the kernel and run all cells.


In [2]:
import importlib

from src.constants import PROCESSED_DIR
# Import pipeline symbols from the module (not the package) and reload so a long-lived
# kernel picks up new names like prune_all after you git pull or edit code.
import src.preprocessing.pipeline as _pipeline
importlib.reload(_pipeline)
from src.preprocessing.pipeline import (
    clean_all,
    flatten_all,
    load_all_raw,
    preprocess_all,
    prune_all,
    reduce_all,
    transform_all,
    write_processed,
)
from src.preprocessing.column_lineage import build_lineage_all, write_column_lineage
from src.preprocessing.config import default_config
from src.spark.session import create_spark_session

In [3]:
# Optional: set *before* create_spark_session. After changing, run spark.stop() then create_spark_session again
# (kernel restart not required). Then re-run pipeline cells — DataFrames from the old session are invalid.
import os
# os.environ["SPARK_LOCAL_DIRS"] = "/var/tmp/spark-scratch"  # only if project/home disk is also quota-limited (default is artifacts/spark_local_scratch)
# os.environ["SPARK_MAX_CORES"] = "1"  # local[1] — lowest CPU load
# os.environ["SPARK_DRIVER_MEMORY"] = "768m"
# os.environ["SPARK_SQL_SHUFFLE_PARTITIONS"] = "4"
# os.environ["YELP_WRITE_PARTITIONS"] = "32"  # more Parquet shards → smaller per-task sorts

In [4]:
spark = create_spark_session("run_preprocessing")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 18:23:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/29 18:23:04 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


**Staged pipeline:** Run the cells below one by one. Stages: 1 load, 2 clean, 3 flatten, 4 prune, 5 transform, 6 reduce, 7 write Parquet + `column_lineage.json`. Each stage logs progress per dataset and a heartbeat every 60s during long steps.

**Resource limits:** `create_spark_session` defaults to `local[2]`, 1g driver memory, bounded shuffle partitions, and **`artifacts/spark_local_scratch`** for Spark shuffle/spill (not `/tmp`). Override **`SPARK_LOCAL_DIRS`** only if that path is still on a small quota. For Parquet writes, set `YELP_WRITE_PARTITIONS`. Launching Jupyter with `nice -n 10` can also keep the desktop more responsive under load.

In [5]:
# Stage 1: Load raw data (progress logged per dataset; heartbeat every 60s during long steps)
raw = load_all_raw(spark)

[19:23:08] Stage 1: Loading raw data (6 datasets)
[19:23:08]   [1/6] business: loading...


[19:23:12]   [1/6] business: done (150346 rows, 4.0s)
[19:23:12]   [2/6] review: loading...


[19:23:24]   [2/6] review: done (6990280 rows, 11.6s)
[19:23:24]   [3/6] user: loading...


[19:23:30]   [3/6] user: done (1987897 rows, 6.5s)
[19:23:30]   [4/6] checkin: loading...


[19:23:31]   [4/6] checkin: done (131930 rows, 0.6s)
[19:23:31]   [5/6] tip: loading...
[19:23:32]   [5/6] tip: done (908915 rows, 0.8s)
[19:23:32]   [6/6] photo: loading...


[19:23:32]   [6/6] photo: done (200100 rows, 0.3s)
[19:23:32] Stage 1: finished.


In [6]:
# Stage 2: Clean (nulls, duplicates)
cleaned = clean_all(spark, raw)

[19:23:32] Stage 2: Cleaning (6 datasets)
[19:23:32]   [1/6] business: cleaning...


[19:23:38]   [1/6] business: done (150346 rows, 5.9s)
[19:23:38]   [2/6] review: cleaning...


[19:24:38]   ... still running: clean review


[19:25:36]   [2/6] review: done (6990280 rows, 117.9s)
[19:25:36]   [3/6] user: cleaning...


[19:26:36]   ... still running: clean user


[19:27:36]   ... still running: clean user


[19:27:54]   [3/6] user: done (1987897 rows, 137.9s)
[19:27:54]   [4/6] checkin: cleaning...


[19:27:58]   [4/6] checkin: done (131930 rows, 3.8s)
[19:27:58]   [5/6] tip: cleaning...


[19:28:04]   [5/6] tip: done (908836 rows, 6.4s)
[19:28:04]   [6/6] photo: cleaning...


[19:28:05]   [6/6] photo: done (200098 rows, 0.6s)
[19:28:05] Stage 2: finished.


In [7]:
# Stage 3: Flatten nested columns (attributes, hours, friends, elite, checkin date)
flattened = flatten_all(cleaned)

[19:28:05] Stage 3: Flattening (6 datasets)
[19:28:05]   [1/6] business: flattening...
[19:28:07]   [1/6] business: done (150346 rows, 1.5s)
[19:28:07]   [2/6] review: flattening...


[19:28:31]   [2/6] review: done (6990280 rows, 24.8s)
[19:28:31]   [3/6] user: flattening...


[19:28:40]   [3/6] user: done (1987897 rows, 8.8s)
[19:28:40]   [4/6] checkin: flattening...


[19:28:43]   [4/6] checkin: done (131930 rows, 2.9s)
[19:28:43]   [5/6] tip: flattening...


[19:28:46]   [5/6] tip: done (908836 rows, 3.1s)
[19:28:46]   [6/6] photo: flattening...


[19:28:47]   [6/6] photo: done (200098 rows, 0.4s)
[19:28:47] Stage 3: finished.


In [8]:
# Stage 4: Prune redundant nested/array columns after flatten
pruned = prune_all(spark, flattened)

[19:28:47] Stage 4: Prune after flatten (6 datasets)
[19:28:47]   [1/6] business: pruning...
[19:28:48]   [1/6] business: done (150346 rows, 0.8s)
[19:28:48]   [2/6] review: pruning...


[19:29:06]   [2/6] review: done (6990280 rows, 18.5s)
[19:29:06]   [3/6] user: pruning...


[19:29:14]   [3/6] user: done (1987897 rows, 8.3s)
[19:29:14]   [4/6] checkin: pruning...


[19:29:17]   [4/6] checkin: done (131930 rows, 2.7s)
[19:29:17]   [5/6] tip: pruning...


[19:29:20]   [5/6] tip: done (908836 rows, 2.7s)
[19:29:20]   [6/6] photo: pruning...


[19:29:20]   [6/6] photo: done (200098 rows, 0.5s)
[19:29:20] Stage 4: finished.


In [9]:
# Stage 5: Transform (scale, parse dates, drop raw date strings after parsing)
transformed = transform_all(spark, pruned)

[19:29:20] Stage 5: Transforming (6 datasets)
[19:29:20]   [1/6] business: transforming...


[19:29:29]   [1/6] business: done (150346 rows, 8.4s)
[19:29:29]   [2/6] review: transforming...


[19:30:29]   ... still running: transform review


[19:31:29]   [2/6] review: done (6990280 rows, 120.4s)
[19:31:29]   [3/6] user: transforming...


[19:32:29]   ... still running: transform user


[19:33:29]   ... still running: transform user


[19:34:34]   [3/6] user: done (1987897 rows, 184.8s)
[19:34:34]   [4/6] checkin: transforming...


[19:34:41]   [4/6] checkin: done (131930 rows, 7.4s)
[19:34:41]   [5/6] tip: transforming...


[19:34:47]   [5/6] tip: done (908836 rows, 5.9s)
[19:34:47]   [6/6] photo: transforming...


[19:34:48]   [6/6] photo: done (200098 rows, 0.5s)
[19:34:48] Stage 5: finished.


In [10]:
# Stage 6: Reduce (optional sampling)
processed = reduce_all(spark, transformed)

[19:34:48] Stage 6: Reduce (6 datasets)
[19:34:48]   [1/6] business: reduce...
[19:34:49]   [1/6] business: done (150346 rows, 0.8s)
[19:34:49]   [2/6] review: reduce...


[19:35:08]   [2/6] review: done (6990280 rows, 18.9s)
[19:35:08]   [3/6] user: reduce...


[19:35:15]   [3/6] user: done (1987897 rows, 7.1s)
[19:35:15]   [4/6] checkin: reduce...


[19:35:18]   [4/6] checkin: done (131930 rows, 2.7s)
[19:35:18]   [5/6] tip: reduce...


[19:35:20]   [5/6] tip: done (908836 rows, 2.8s)
[19:35:20]   [6/6] photo: reduce...


[19:35:21]   [6/6] photo: done (200098 rows, 0.4s)
[19:35:21] Stage 6: finished.


In [11]:
# Stage 7: Write Parquet and column lineage JSON (artifacts/processed/column_lineage.json)
write_processed(processed)
cols_cleaned = {n: list(df.columns) for n, df in cleaned.items()}
cols_flat = {n: list(df.columns) for n, df in flattened.items()}
cols_pruned = {n: list(df.columns) for n, df in pruned.items()}
cols_final = {n: list(df.columns) for n, df in processed.items()}
lineage = build_lineage_all(cols_cleaned, cols_flat, cols_pruned, cols_final, default_config())
write_column_lineage(lineage)

[19:35:21] Writing Parquet (6 datasets) to /home/dash/University/term-6/big-data/yelp-spark-project/artifacts/processed (16 partitions each)
[19:35:21]   [1/6] business: writing...


26/03/29 18:35:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[19:35:32]   [1/6] business: done (10.8s)
[19:35:32]   [2/6] review: writing...


[19:44:18]   [2/6] review: done (525.9s)
[19:44:18]   [3/6] user: writing...


[19:45:00]   [3/6] user: done (42.3s)
[19:45:00]   [4/6] checkin: writing...


[19:45:08]   [4/6] checkin: done (7.9s)
[19:45:08]   [5/6] tip: writing...


[19:45:16]   [5/6] tip: done (8.2s)
[19:45:16]   [6/6] photo: writing...


[19:45:18]   [6/6] photo: done (1.9s)
[19:45:18] Writing: finished.


PosixPath('/home/dash/University/term-6/big-data/yelp-spark-project/artifacts/processed/column_lineage.json')